In [ ]:
# ==============================================================================
# Google Colab: EEG 情绪识别快速复现 (基于 Kaggle 公开数据集)
# 数据集: EEG Brainwave Dataset: Feeling Emotions
# ==============================================================================

# 1. 安装必要的库
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q
!pip install kaggle -q

# 2. 导入库
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from google.colab import files

# 3. 上传 Kaggle API 密钥
print("--- 请上传您的 kaggle.json 文件 (从 Kaggle 账号设置中生成) ---")
uploaded = files.upload()
for fn in uploaded.keys():
    print(f'成功上传: {fn}')

# 配置 Kaggle 路径
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 4. 下载数据集
print("\n--- 正在下载数据集... ---")
!kaggle datasets download -d birdy654/eeg-brainwave-dataset-feeling-emotions
!unzip -q eeg-brainwave-dataset-feeling-emotions.zip -d ./data/
print("数据集下载完成！")

# 5. 数据加载与预处理
print("\n--- 正在预处理数据... ---")
df = pd.read_csv('./data/emotions.csv')

# 分离特征 (X) 和标签 (y)
X = df.drop('label', axis=1).values
y = df['label'].values

# 编码标签: NEGATIVE, NEUTRAL, POSITIVE -> 0, 1, 2
le = LabelEncoder()
y = le.fit_transform(y)
print(f"标签映射: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# 标准化特征
scaler = StandardScaler()
X = scaler.fit_transform(X)

# 划分数据集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 转换为 PyTorch 张量 (增加通道维度)
X_train = torch.FloatTensor(X_train).unsqueeze(1)
X_test = torch.FloatTensor(X_test).unsqueeze(1)
y_train = torch.LongTensor(y_train)
y_test = torch.LongTensor(y_test)

print(f"训练集形状: {X_train.shape}")
print(f"测试集形状: {X_test.shape}")

# 6. 定义 CNN 模型 (简化版 DGCNN 架构思想)
class EEG_CNN(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(EEG_CNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.MaxPool1d(2)
        )

        # 自动计算全连接层输入维度
        with torch.no_grad():
            dummy = torch.randn(1, 1, input_dim)
            dummy = self.conv_layers(dummy)
            fc_in_dim = dummy.view(1, -1).size(1)

        self.fc_layers = nn.Sequential(
            nn.Linear(fc_in_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x

# 7. 训练配置
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = EEG_CNN(input_dim=X_train.shape[2], num_classes=3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001)

# 数据加载器
batch_size = 32
train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
test_dataset = torch.utils.data.TensorDataset(X_test, y_test)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"\n--- 开始训练 (使用设备: {device}) ---")

# 8. 训练循环
epochs = 40
best_acc = 0.0

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    # 训练阶段
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)

    # 验证阶段
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)

    # 保存最佳模型
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), 'eeg_emotion_best.pth')

    print(f"Epoch [{epoch+1:2d}/{epochs}] -> Loss: {epoch_loss:.4f} | Acc: {acc:.4f} | Best: {best_acc:.4f}")

print("\n--- 训练完成！---")

# 9. 最终评估
model.load_state_dict(torch.load('eeg_emotion_best.pth'))
model.eval()
all_preds = []
with torch.no_grad():
    for inputs, _ in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())

print("\n分类报告:")
print(classification_report(y_test, all_preds, target_names=le.classes_))

--- 请上传您的 kaggle.json 文件 (从 Kaggle 账号设置中生成) ---


cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory

--- 正在下载数据集... ---
Dataset URL: https://www.kaggle.com/datasets/birdy654/eeg-brainwave-dataset-feeling-emotions
License(s): copyright-authors
100% 11.9M/11.9M [00:00<00:00, 83.2MB/s]

数据集下载完成！

--- 正在预处理数据... ---
标签映射: {'NEGATIVE': np.int64(0), 'NEUTRAL': np.int64(1), 'POSITIVE': np.int64(2)}
训练集形状: torch.Size([1705, 1, 2548])
测试集形状: torch.Size([427, 1, 2548])

--- 开始训练 (使用设备: cuda) ---
Epoch [ 1/40] -> Loss: 1.4247 | Acc: 0.9625 | Best: 0.9625
Epoch [ 2/40] -> Loss: 0.3224 | Acc: 0.9813 | Best: 0.9813
Epoch [ 3/40] -> Loss: 0.3260 | Acc: 0.9742 | Best: 0.9813
Epoch [ 4/40] -> Loss: 0.1311 | Acc: 0.9555 | Best: 0.9813
Epoch [ 5/40] -> Loss: 0.1201 | Acc: 0.9930 | Best: 0.9930
Epoch [ 6/40] -> Loss: 0.1051 | Acc: 0.9813 | Best: 0.9930
Epoch [ 7/40] -> Loss: 0.0383 | Acc: 0.9883 | Best: 0.9930
Epoch [ 8/40] -> Loss: 0.0152 | Acc: 0.9930 | Best: 0.9930
Epoc